# Strong Conditional Diffusion Model Training

**Enhanced Architecture with Cross-Attention + Classifier-Free Guidance**

This notebook implements a **proven conditional diffusion architecture** with:
- Cross-attention conditioning in every transformer layer
- Classifier-free guidance with condition dropout
- Multi-scale condition embedding
- CFG inference for strong conditional control

**Expected Performance**: +/-0.02-0.05 QED accuracy (vs current +/-0.15)

## Setup & Imports

In [ ]:
!pip install rdkit selfies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Core libraries
import os
import json
import time
import math
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from typing import Dict, List, Optional, Tuple, Any
import matplotlib.pyplot as plt

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

# Chemistry & ML
from rdkit import Chem
from rdkit.Chem import Descriptors, QED
import selfies as sf

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
set_seed(42)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name()}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Configuration

In [ ]:
class Config:
    """Training configuration for strong conditional diffusion."""
    
    # Model architecture
    VOCAB_SIZE = 135
    MAX_LEN = 96
    D_MODEL = 256
    N_HEADS = 8
    N_LAYERS = 6
    D_FF = 1024
    DROPOUT = 0.1
    N_CONDITIONS = 8
    COND_DROPOUT = 0.2  # Increased from 0.15 for stronger CFG effect
    
    # Conditioning control
    COND_STRENGTH_ALPHA = 0.5
    N_COND_TOKENS = 6
    
    # Training hyperparameters  
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    EPOCHS = 100
    WARMUP_EPOCHS = 10
    
    # Diffusion parameters
    N_TIMESTEPS = 1000
    BETA_START = 1e-4
    BETA_END = 2e-2
    
    # CFG parameters
    CFG_SCALE = 7.5
    
    # Data paths (Google Colab / Drive)
    DATA_PATH = "/content/drive/MyDrive/Datasets/Drug Discovery/molecules_200k_selfies_clean.csv"
    TOKENIZER_PATH = "/content/drive/MyDrive/Datasets/Drug Discovery/selfies_tokenizer.json"
    MODEL_DIR = "/content/drive/MyDrive/Models/Conditional_New_Arch"
    
    # Data filtering (Stage 2: drug-like molecules only)
    FILTER_MIN_QED = 0.5  # Filter to QED >= 0.5 (~161k molecules, 80.69%)
    
    # Logging
    SAVE_EVERY = 10
    LOG_EVERY = 100

config = Config()
os.makedirs(config.MODEL_DIR, exist_ok=True)
print(f"Model directory: {config.MODEL_DIR}")

## Data Loading & Preprocessing

In [ ]:
def load_tokenizer():
    """Load SELFIES tokenizer."""
    with open(config.TOKENIZER_PATH, 'r') as f:
        tokenizer_data = json.load(f)
    
    # Handle different tokenizer formats
    if 'token_to_id' in tokenizer_data:
        token_to_id = tokenizer_data['token_to_id']
        id_to_token = {v: k for k, v in token_to_id.items()}
    elif 'vocab' in tokenizer_data:
        # Build from vocab list (index = token ID)
        vocab = tokenizer_data['vocab']
        token_to_id = {token: idx for idx, token in enumerate(vocab)}
        id_to_token = {idx: token for idx, token in enumerate(vocab)}
    else:
        raise KeyError("Tokenizer must have 'token_to_id' or 'vocab' key")
    
    # Get special tokens - convert to IDs if stored as token strings
    raw_special = tokenizer_data.get('special_tokens', {})
    special_tokens = {}
    for name, val in raw_special.items():
        if isinstance(val, str):
            # Value is token string, look up its ID
            special_tokens[name] = token_to_id.get(val, token_to_id.get(f'[{name}]', 0))
        else:
            special_tokens[name] = val
    
    print(f"Loaded tokenizer: {len(token_to_id)} tokens")
    print(f"Special tokens: {special_tokens}")
    
    return token_to_id, id_to_token, special_tokens

# The 8 conditioning features (matching IMPLEMENTATION_STATUS.md Stage 2)
CONDITION_FEATURES = ['qed', 'sa_score', 'mw', 'alogp', 'hbd', 'hba', 'psa', 'num_rings']

def get_condition_stats():
    """Default molecular property statistics for normalization."""
    # These are typical values for drug-like molecules
    # Will be overwritten by compute_condition_stats() after data loading
    return {
        'qed': {'mean': 0.5, 'std': 0.2},
        'sa_score': {'mean': 2.8, 'std': 0.8},
        'mw': {'mean': 350.0, 'std': 100.0}, 
        'alogp': {'mean': 2.5, 'std': 1.5},
        'hbd': {'mean': 1.5, 'std': 1.2},
        'hba': {'mean': 4.0, 'std': 2.0},
        'psa': {'mean': 70.0, 'std': 30.0},
        'num_rings': {'mean': 2.5, 'std': 1.0}
    }

def compute_condition_stats(properties_list: List[Dict]) -> Dict:
    """
    Compute condition statistics from actual training data.
    
    CRITICAL: Use these stats for both training and inference!
    Save them with the model checkpoint for consistency.
    """
    stats = {}
    for feat in CONDITION_FEATURES:
        vals = [p[feat] for p in properties_list if feat in p and p[feat] is not None]
        if vals:
            stats[feat] = {'mean': np.mean(vals), 'std': np.std(vals) + 1e-8}
        else:
            # Fallback to defaults
            stats[feat] = get_condition_stats()[feat]
    
    print("Computed condition stats from training data:")
    for prop, s in stats.items():
        print(f"   {prop}: mean={s['mean']:.3f}, std={s['std']:.3f}")
    
    return stats

# Load data components
token_to_id, id_to_token, special_tokens = load_tokenizer()
cond_stats = None  # MUST be computed from TRAIN split only - no defaults!

# Unified UNK ID (check special_tokens first, then vocab)
UNK_ID = special_tokens.get('UNK', token_to_id.get('[UNK]', None))

# Helper functions
def smiles_to_selfies(smiles: str) -> str:
    """Convert SMILES to SELFIES safely."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return sf.encoder(smiles)
    except:
        return None

def calculate_properties(smiles: str) -> Optional[Dict[str, float]]:
    """Calculate all 8 molecular properties for conditioning."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        from rdkit.Chem import rdMolDescriptors

        # SA score (optional)
        sa_score = None
        try:
            from rdkit.Contrib.SA_Score import sascorer
            sa_score = float(sascorer.calculateScore(mol))
        except Exception:
            sa_score = 3.0  # neutral fallback (won't crash)

        props = {
            'qed': float(QED.qed(mol)),
            'sa_score': float(sa_score),
            'mw': float(Descriptors.MolWt(mol)),
            'alogp': float(Descriptors.MolLogP(mol)),
            'hbd': float(Descriptors.NumHDonors(mol)),
            'hba': float(Descriptors.NumHAcceptors(mol)),
            'psa': float(Descriptors.TPSA(mol)),
            'num_rings': float(rdMolDescriptors.CalcNumRings(mol)),
        }

        return props
    except Exception:
        return None

In [ ]:
# ============================================
# CRITICAL: VERIFY TOKENIZER SPECIAL TOKEN IDs
# ============================================
# Ensures PAD/BOS/EOS/UNK are correct before training

print("="*70)
print("TOKENIZER VERIFICATION")
print("="*70)

print("\n1. Special Token IDs:")
issues = []

# Check that essential tokens exist and are distinct
required_tokens = ['PAD', 'BOS', 'EOS']
found_ids = {}

for name in required_tokens:
    token_id = special_tokens.get(name, None)
    if token_id is None:
        print(f"   ❌ {name}: NOT DEFINED (required!)")
        issues.append(f"{name} missing")
    else:
        found_ids[name] = token_id
        print(f"   ✅ {name}: ID={token_id}")

# Check UNK (optional) - now using global UNK_ID
if UNK_ID is None:
    print(f"   ⚠️  UNK: Not defined (will skip unknown tokens)")
else:
    found_ids['UNK'] = UNK_ID
    print(f"   ✅ UNK: ID={UNK_ID}")

# Check IDs are distinct
if len(found_ids) > 0:
    id_values = list(found_ids.values())
    if len(id_values) != len(set(id_values)):
        print(f"   ❌ WARNING: Duplicate special token IDs detected!")
        issues.append("Duplicate IDs")

print("\n2. Token Coverage Check:")
# Check if expected tokens exist in vocabulary
for name, token_id in special_tokens.items():
    if token_id in id_to_token:
        token_str = id_to_token[token_id]
        print(f"   ✅ {name} (ID {token_id}) → '{token_str}'")
    else:
        print(f"   ❌ {name} (ID {token_id}) not in vocabulary!")
        issues.append(f"{name} not in vocab")

print("\n3. SA Score Import Check:")
try:
    test_smiles = "CCO"  # Ethanol
    test_props = calculate_properties(test_smiles)
    
    if test_props is None:
        print("   ❌ FAIL: calculate_properties returned None")
        issues.append("SA score calculation failed")
    elif 'sa_score' not in test_props:
        print("   ❌ FAIL: sa_score missing from properties")
        issues.append("SA score missing")
    else:
        sa_val = test_props['sa_score']
        print(f"   ✅ PASS: SA score calculated ({sa_val:.2f})")
        print(f"   All 8 properties: {list(test_props.keys())}")
except Exception as e:
    print(f"   ❌ FAIL: {e}")
    issues.append(f"SA score error: {e}")

print("\n" + "="*70)
if issues:
    print("⚠️  WARNINGS FOUND:")
    for issue in issues:
        print(f"   - {issue}")
    print("\nThese issues may cause training failures!")
else:
    print("✅ ALL CHECKS PASSED - Ready for training")
print("="*70)

In [ ]:
def prepare_conditions(properties: Dict[str, float], target_only: str = None) -> torch.Tensor:
    """
    Prepare normalized condition vector for all 8 features.
    
    Args:
        properties: Dict of property values
        target_only: If set (e.g. 'qed'), only that property is set, others are neutral (0)
    
    Features (8): qed, sa_score, mw, alogp, hbd, hba, psa, num_rings
    """
    if cond_stats is None:
        raise RuntimeError("cond_stats not computed! Call compute_condition_stats() on train data first.")
    
    normalized = []
    
    for prop in CONDITION_FEATURES:
        if target_only and prop != target_only:
            # Neutral: 0 in normalized space = mean in raw space
            norm_value = 0.0
        else:
            value = properties.get(prop, 0.0)
            
            if prop in cond_stats:
                mean = cond_stats[prop]['mean']
                std = cond_stats[prop]['std']
                norm_value = (value - mean) / (std + 1e-8)
                norm_value = np.clip(norm_value, -3.0, 3.0)  # Clamp to +/-3 sigma
            else:
                norm_value = value
            
        normalized.append(norm_value)
    
    return torch.tensor(normalized, dtype=torch.float32)

def tokenize_selfies(selfies: str) -> List[int]:
    """Convert SELFIES string to token IDs."""
    if selfies is None:
        return None
        
    try:
        tokens = list(sf.split_selfies(selfies))
        token_ids = [special_tokens.get('BOS', 1)]
        
        for token in tokens:
            if token in token_to_id:
                token_ids.append(token_to_id[token])
            else:
                # Use unified UNK_ID
                if UNK_ID is not None:
                    token_ids.append(UNK_ID)
                # else: skip the unknown token entirely
        
        token_ids.append(special_tokens.get('EOS', 2))
        
        # Pad or truncate to max_len
        if len(token_ids) > config.MAX_LEN:
            token_ids = token_ids[:config.MAX_LEN]
        else:
            pad_id = special_tokens.get('PAD', 0)
            token_ids.extend([pad_id] * (config.MAX_LEN - len(token_ids)))
            
        return token_ids
    except:
        return None

print(f"Data preprocessing functions ready")
print(f"Conditioning on {len(CONDITION_FEATURES)} features: {CONDITION_FEATURES}")

print(f"Data preprocessing functions ready")

## Strong Conditional Diffusion Architecture

In [ ]:
import math

class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal position embeddings for time steps."""
    
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        
    def forward(self, time: torch.Tensor) -> torch.Tensor:
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class CrossAttentionBlock(nn.Module):
    """Cross-attention between sequence and conditions."""
    
    def __init__(self, d_model: int, n_heads: int, cond_dim: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        
        # Self-attention for sequence tokens
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.self_norm = nn.LayerNorm(d_model)
        
        # Cross-attention to conditions  
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_norm = nn.LayerNorm(d_model)
        
        # Residual strength control (prevents overpowering)
        self.cond_strength = config.COND_STRENGTH_ALPHA
        
        # FiLM layers: condition-based scale/shift (forces non-ignorable conditioning)
        self.film_scale = nn.Linear(d_model, d_model)
        self.film_shift = nn.Linear(d_model, d_model)
        
        # Feed-forward network
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model)
        )
        self.ff_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, cond: torch.Tensor, 
                key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Self-attention
        attn_out, _ = self.self_attn(x, x, x, key_padding_mask=key_padding_mask)
        x = self.self_norm(x + self.dropout(attn_out))
        
        # Cross-attention to conditions
        cross_out, _ = self.cross_attn(x, cond, cond)
        x = self.cross_norm(x + self.cond_strength * self.dropout(cross_out))
        
        # FiLM conditioning: scale and shift based on condition
        # Pool condition info (mean across condition tokens)
        cond_pooled = cond.mean(dim=1)  # [B, d_model]
        gamma = self.film_scale(cond_pooled).unsqueeze(1)  # [B, 1, d_model]
        beta = self.film_shift(cond_pooled).unsqueeze(1)   # [B, 1, d_model]
        
        # Apply FiLM: x = gamma * x + beta
        # This ensures conditioning affects every activation
        x = gamma * x + beta
        
        # Feed-forward network
        ff_out = self.ff(x)
        x = self.ff_norm(x + self.dropout(ff_out))
        
        return x

print("Cross-attention block defined")

In [ ]:
class StrongConditionalDiffusion(nn.Module):
    """Strong Conditional Diffusion with cross-attention and CFG support."""
    
    def __init__(self, vocab_size: int, max_len: int = 96, d_model: int = 256,
                 n_heads: int = 8, n_layers: int = 6, dropout: float = 0.1,
                 n_conditions: int = 8, cond_dropout: float = 0.15):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.d_model = d_model
        self.n_conditions = n_conditions
        self.cond_dropout = cond_dropout
        
        # Token and position embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        
        # Time embedding
        self.time_emb = nn.Sequential(
            SinusoidalPositionEmbeddings(d_model),
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Linear(d_model * 2, d_model)
        )
        
        # Condition embedding
        self.cond_embed = nn.Sequential(
            nn.Linear(n_conditions, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, d_model),
            nn.LayerNorm(d_model)
        )
        
        # Learnable condition tokens
        self.cond_tokens = nn.Parameter(torch.randn(config.N_COND_TOKENS, d_model))
        
        # Learned null tokens for proper CFG (not zeros)
        self.null_cond_tokens = nn.Parameter(torch.randn(config.N_COND_TOKENS, d_model))
        
        # Cross-attention transformer layers  
        self.layers = nn.ModuleList([
            CrossAttentionBlock(d_model, n_heads, d_model, dropout)
            for _ in range(n_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        
        # Property prediction head
        self.property_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, n_conditions)
        )
        
        self._init_weights()
        
    def _init_weights(self):
        """Xavier initialization for stable training."""
        for name, p in self.named_parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
            elif 'bias' in name:
                nn.init.constant_(p, 0)
                
    def _prepare_condition_tokens(self, cond: Optional[torch.Tensor], batch_size: int,
                                   use_cond: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Create condition representation with learnable tokens.
        
        Args:
            cond: [B, n_conditions] or None
            batch_size: B
            use_cond: [B] boolean mask - if provided, samples with False use null tokens
        """
        if cond is None:
            # All unconditional: use learned null tokens
            return self.null_cond_tokens.unsqueeze(0).expand(batch_size, -1, -1)
        
        # Embed conditions
        cond_emb = self.cond_embed(cond)  # [B, d_model]
        
        # Expand condition tokens for batch
        cond_tokens = self.cond_tokens.unsqueeze(0).expand(batch_size, -1, -1)
        
        # Inject condition info into each token
        cond_tokens = cond_tokens + cond_emb.unsqueeze(1)
        
        # Handle per-sample CFG dropout via use_cond mask
        if use_cond is not None:
            # Expand null tokens for batch
            null_tokens = self.null_cond_tokens.unsqueeze(0).expand(batch_size, -1, -1)
            
            # Where use_cond is False, use null tokens
            use_cond_expanded = use_cond.view(batch_size, 1, 1).expand(-1, self.cond_tokens.size(0), -1)
            cond_tokens = torch.where(use_cond_expanded, cond_tokens, null_tokens)
        
        return cond_tokens
    
    def forward(self, x: torch.Tensor, t: torch.Tensor, 
                cond: Optional[torch.Tensor] = None,
                use_cond: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass with CFG support.
        
        Args:
            x: [B, seq_len] token sequences
            t: [B] diffusion timesteps
            cond: [B, n_conditions] conditions (None for all unconditional)
            use_cond: [B] boolean mask for per-sample CFG dropout
            
        Returns:
            logits: [B, seq_len, vocab_size] 
            properties: [B, n_conditions] predicted properties
        """
        B, L = x.shape
        device = x.device
        
        # Token + position embeddings
        pos = torch.arange(L, device=device).unsqueeze(0).expand(B, -1)
        h = self.token_emb(x) + self.pos_emb(pos)
        
        # Add time embedding
        t_emb = self.time_emb(t.float())
        h = h + t_emb.unsqueeze(1)
        
        h = self.dropout(h)
        
        # Prepare condition tokens (handles null per sample based on use_cond)
        cond_tokens = self._prepare_condition_tokens(cond, B, use_cond)
        
        # Key padding mask
        key_padding_mask = (x == special_tokens.get('PAD', 0))
        
        # Cross-attention layers
        for layer in self.layers:
            h = layer(h, cond_tokens, key_padding_mask)
        
        # Output
        h = self.norm(h)
        logits = self.output(h)
        
        # Property prediction from pooled representation
        h_pooled = h.mean(dim=1)
        properties = self.property_head(h_pooled)
        
        return logits, properties
    
    def cfg_forward(self, x: torch.Tensor, t: torch.Tensor, 
                    cond: torch.Tensor, cfg_scale: float = 7.5) -> torch.Tensor:
        """Classifier-Free Guidance inference."""
        # Conditional prediction
        cond_logits, _ = self.forward(x, t, cond)
        
        # Unconditional prediction (uses learned null tokens)
        uncond_logits, _ = self.forward(x, t, None)
        
        # CFG: amplify the difference
        guided_logits = uncond_logits + cfg_scale * (cond_logits - uncond_logits)
        
        return guided_logits

print("StrongConditionalDiffusion model defined")

## Dataset & DataLoader

In [ ]:
class MolecularDataset(Dataset):
    """Dataset for molecular SELFIES with properties."""
    
    def __init__(self, selfies_list: List[str], properties_list: List[Dict], 
                 apply_cond_dropout: bool = True):
        self.selfies_list = selfies_list
        self.properties_list = properties_list
        self.apply_cond_dropout = apply_cond_dropout
        
        # Pre-tokenize all sequences
        print("Tokenizing SELFIES...")
        self.tokenized_data = []
        self.condition_data = []
        
        for selfies, props in tqdm(zip(selfies_list, properties_list), total=len(selfies_list)):
            tokens = tokenize_selfies(selfies)
            cond_vec = prepare_conditions(props)
            
            if tokens is not None:
                self.tokenized_data.append(tokens)
                self.condition_data.append(cond_vec)
        
        print(f"Dataset ready: {len(self.tokenized_data)} samples")
        
    def __len__(self):
        return len(self.tokenized_data)
    
    def __getitem__(self, idx):
        tokens = torch.tensor(self.tokenized_data[idx], dtype=torch.long)
        conditions = self.condition_data[idx].clone()
        
        # CFG dropout: return mask indicating whether to use conditions
        use_cond = True
        if self.apply_cond_dropout:
            if torch.rand(1).item() < config.COND_DROPOUT:
                use_cond = False  # Model will use learned null tokens
        
        return {
            'tokens': tokens,
            'conditions': conditions,
            'use_cond': torch.tensor(use_cond, dtype=torch.bool)  # CFG mask
        }

def load_real_dataset():
    """Load real dataset with QED filtering and proper train/val split."""
    from sklearn.model_selection import train_test_split
    
    print("="*70)
    print("LOADING REAL DATASET (STAGE 2: CONDITIONAL TRAINING)")
    print("="*70)
    
    # 1. Load CSV
    print(f"\nLoading data from {config.DATA_PATH}...")
    df = pd.read_csv(config.DATA_PATH)
    print(f"Loaded: {len(df):,} molecules")
    
    # 2. Filter to QED >= 0.5 (Stage 2: drug-like molecules only)
    if config.FILTER_MIN_QED is not None:
        df_filtered = df[df['qed'] >= config.FILTER_MIN_QED].reset_index(drop=True)
        print(f"\nFiltered to QED >= {config.FILTER_MIN_QED}:")
        print(f"   Kept: {len(df_filtered):,} molecules ({len(df_filtered)/len(df)*100:.1f}%)")
        print(f"   QED range: [{df_filtered['qed'].min():.3f}, {df_filtered['qed'].max():.3f}]")
    else:
        df_filtered = df
        print(f"\nUsing all molecules (no QED filter)")
    
    # 3. Pre-validate: create (selfies, props) pairs and test tokenization
    print(f"\nPre-validating tokenization...")
    valid_data = []
    
    for idx, row in tqdm(df_filtered.iterrows(), total=len(df_filtered), desc="Validating"):
        selfies = row['selfies']
        props = {feat: row[feat] for feat in CONDITION_FEATURES}
        
        # Test tokenization
        tokens = tokenize_selfies(selfies)
        if tokens is not None:
            valid_data.append((selfies, props))
    
    print(f"Valid tokenizable: {len(valid_data):,} molecules ({len(valid_data)/len(df_filtered)*100:.1f}%)")
    
    if len(valid_data) == 0:
        raise ValueError("No valid molecules after tokenization! Check tokenizer and data.")
    
    # 4. Train/val split (90/10) on VALID data
    print(f"\nSplitting train/val (90/10)...")
    train_data, val_data = train_test_split(
        valid_data, 
        test_size=0.1, 
        random_state=42,
        shuffle=True
    )
    
    print(f"Train: {len(train_data):,} molecules")
    print(f"Val:   {len(val_data):,} molecules")
    
    # 5. Extract components
    train_selfies, train_props = zip(*train_data)
    val_selfies, val_props = zip(*val_data)
    
    # Convert to lists
    train_selfies = list(train_selfies)
    train_props = list(train_props)
    val_selfies = list(val_selfies)
    val_props = list(val_props)
    
    return train_selfies, train_props, val_selfies, val_props

# ============================================================================
# DATA LOADING: Load real dataset with QED filtering
# ============================================================================

print("\nLoading REAL dataset for training...")
train_selfies, train_props, val_selfies, val_props = load_real_dataset()

print("\n" + "="*70)
print("COMPUTING CONDITION STATISTICS (FROM TRAIN SPLIT ONLY)")
print("="*70)

# CRITICAL: Compute stats from TRAIN split ONLY
cond_stats = compute_condition_stats(train_props)

# Save stats to disk (required for inference!)
stats_path = os.path.join(config.MODEL_DIR, 'condition_stats.json')
with open(stats_path, 'w') as f:
    json.dump(cond_stats, f, indent=2)

print(f"\nCondition stats saved to: {stats_path}")

print("\n" + "="*70)
print("CREATING DATASETS")
print("="*70)

# Create datasets (both use same train stats for normalization)
train_dataset = MolecularDataset(train_selfies, train_props, apply_cond_dropout=True)
val_dataset = MolecularDataset(val_selfies, val_props, apply_cond_dropout=False)

print(f"\nTrain dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")

# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print("\n" + "="*70)

## Training Setup

In [ ]:
# ============================================================================
# SANITY CHECK 1: Verify Special Token IDs
# ============================================================================
print("="*70)
print("CHECK 1: Special Token IDs")
print("="*70)
print(f"special_tokens dict: {special_tokens}")
print(f"\nChecking UNK token (using global UNK_ID):")
print(f"   UNK_ID = {UNK_ID}")

if UNK_ID is None:
    print(f"   ✅ No UNK token (tokenizer handles all tokens)")
    print(f"   Sampler will block: [PAD, BOS, EOS]")
else:
    print(f"   ✅ UNK token defined (ID={UNK_ID})")
    print(f"   Sampler will block: [PAD, BOS, EOS, UNK]")

print("="*70)

print(f"\nVerified tokens:")
print(f"   PAD = {special_tokens.get('PAD', 0)}")
print(f"   BOS = {special_tokens.get('BOS', 1)}")
print(f"   EOS = {special_tokens.get('EOS', 2)}")
print(f"   UNK = {UNK_ID if UNK_ID is not None else 'N/A'}")

In [ ]:
# ============================================================================
# SANITY CHECK 2: Verify Logit Masking in Generate Function
# ============================================================================
print("\n" + "="*70)
print("CHECK 2: Logit Masking in Sampler")
print("="*70)
print("⚠️  NOTE: This check requires generate_with_cfg to be defined first.")
print("   Run this cell again after executing the CFG inference section.\n")

import inspect

# Check if function is defined
try:
    gen_source = inspect.getsource(generate_with_cfg)
    
    # Check that masking happens before both branches
    has_masking = "guided_logits[:, :, forbidden] = -1e9" in gen_source
    masking_before_if = gen_source.index("guided_logits[:, :, forbidden]") < gen_source.index("if i < len(timesteps)")
    has_softmax = "F.softmax(guided_logits" in gen_source
    has_argmax = "torch.argmax(guided_logits" in gen_source
    
    print(f"✅ Masking code present: {has_masking}")
    print(f"✅ Masking before if/else split: {masking_before_if}")
    print(f"✅ Softmax sampling uses masked logits: {has_softmax}")
    print(f"✅ Argmax final step uses masked logits: {has_argmax}")
    
    if all([has_masking, masking_before_if, has_softmax, has_argmax]):
        print(f"\n✅ PASS: Masking applied to both sampling and argmax branches")
    else:
        print(f"\n❌ FAIL: Masking not correctly applied!")
        
except NameError:
    print("⏭️  SKIPPED: generate_with_cfg not defined yet")
    print("   Define it in the 'Classifier-Free Guidance Inference' section")
    print("   then re-run this cell")
    
print("="*70)

In [ ]:
# ============================================================================
# SANITY CHECK 3: Smoke Test - Untrained Generation
# ============================================================================
print("\n" + "="*70)
print("CHECK 3: Smoke Test (Untrained Model)")
print("="*70)
print("⚠️  NOTE: This check requires generate_with_cfg and tokens_to_molecules.")
print("   Run this cell after executing the CFG inference section.\n")

try:
    # Check if required functions exist
    _ = generate_with_cfg
    _ = tokens_to_molecules
    
    # Initialize a fresh small model for testing (don't wait for training)
    test_model = StrongConditionalDiffusion(
        vocab_size=config.VOCAB_SIZE,
        max_len=config.MAX_LEN,
        d_model=config.D_MODEL,
        n_heads=config.N_HEADS,
        n_layers=config.N_LAYERS,
        dropout=config.DROPOUT,
        n_conditions=config.N_CONDITIONS,
        cond_dropout=config.COND_DROPOUT
    ).to(device)
    
    print("Testing generation with untrained model (should be structurally valid)...")
    print("   num_samples=4, num_steps=5, cfg_scale=7.5\n")
    
    # Generate
    test_conditions = {'qed': 0.7, 'mw': 300, 'alogp': 2.0}
    try:
        generated_tokens = generate_with_cfg(
            test_model, 
            test_conditions, 
            num_samples=4, 
            cfg_scale=7.5, 
            max_len=config.MAX_LEN,
            num_steps=5  # Fast test
        )
        
        print("\n✅ Generation completed without crash")
        
        # Check structural validity
        pad_id = special_tokens.get('PAD', 0)
        bos_id = special_tokens.get('BOS', 1)
        
        # Build forbidden list (use global UNK_ID)
        forbidden_tokens = [pad_id, bos_id]
        if UNK_ID is not None:
            forbidden_tokens.append(UNK_ID)
        
        problems = []
        
        for idx, seq in enumerate(generated_tokens):
            # Check BOS at position 0
            if seq[0].item() != bos_id:
                problems.append(f"Sample {idx}: BOS not at position 0 (got {seq[0].item()})")
            
            # Check no forbidden tokens after position 0
            for pos in range(1, len(seq)):
                token_id = seq[pos].item()
                if token_id in forbidden_tokens:
                    problems.append(f"Sample {idx}, pos {pos}: Forbidden token {token_id}")
                    break
        
        if problems:
            print("❌ FAIL: Structural issues found:")
            for p in problems[:5]:  # Show first 5
                print(f"   {p}")
        else:
            forbidden_str = "PAD/BOS/UNK" if UNK_ID is not None else "PAD/BOS"
            print("✅ PASS: All samples structurally valid")
            print(f"   - BOS always at position 0")
            print(f"   - No {forbidden_str} after position 0")
        
        # Try decoding (should not crash even if nonsense)
        print("\n✅ Decoding test (should not crash):")
        try:
            test_mols = tokens_to_molecules(generated_tokens[:2])  # Just test 2
            valid_count = sum(1 for m in test_mols if m.get('valid', False))
            print(f"   Decoded 2 samples: {valid_count} valid (untrained, so 0-1 is normal)")
        except Exception as e:
            print(f"   ❌ Decoding crashed: {e}")
        
    except Exception as e:
        print(f"❌ FAIL: Generation crashed: {e}")
        import traceback
        traceback.print_exc()
    
    # Cleanup
    del test_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
except NameError as e:
    print(f"⏭️  SKIPPED: Required function not defined yet: {e}")
    print("   Define generate_with_cfg and tokens_to_molecules first")
    print("   (in the 'Classifier-Free Guidance Inference' section)")
    print("   then re-run this cell")

print("="*70)

In [ ]:
# Initialize model
model = StrongConditionalDiffusion(
    vocab_size=config.VOCAB_SIZE,
    max_len=config.MAX_LEN,
    d_model=config.D_MODEL,
    n_heads=config.N_HEADS,
    n_layers=config.N_LAYERS,
    dropout=config.DROPOUT,
    n_conditions=config.N_CONDITIONS,
    cond_dropout=config.COND_DROPOUT
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model initialized!")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / 1024**2:.1f} MB")

# Optimizer & scheduler with warmup
optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

# Warmup + Cosine scheduler (respects config.WARMUP_EPOCHS)
def lr_lambda(epoch):
    if epoch < config.WARMUP_EPOCHS:
        # Linear warmup
        return (epoch + 1) / config.WARMUP_EPOCHS
    # Cosine annealing after warmup
    progress = (epoch - config.WARMUP_EPOCHS) / max(1, config.EPOCHS - config.WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

# Loss function
criterion = nn.CrossEntropyLoss(ignore_index=special_tokens.get('PAD', 0))
property_criterion = nn.MSELoss()

print(f"Optimizer: AdamW (lr={config.LEARNING_RATE}, wd={config.WEIGHT_DECAY})")
print(f"Scheduler: Warmup ({config.WARMUP_EPOCHS} epochs) + CosineAnnealing")
print(f"Loss: CrossEntropyLoss + MSE for properties")

In [ ]:
def add_noise_to_tokens(clean_tokens, timesteps):
    """Add noise to tokens for diffusion training (preserves special tokens)."""
    B, L = clean_tokens.shape
    device = clean_tokens.device
    
    # Get special token IDs
    pad_id = special_tokens.get('PAD', 0)
    bos_id = special_tokens.get('BOS', 1)
    eos_id = special_tokens.get('EOS', 2)
    
    # Build set of special tokens to never noise
    special_set = {pad_id, bos_id, eos_id}
    if UNK_ID is not None:
        special_set.add(UNK_ID)
    
    # CRITICAL: Never noise special tokens (PAD/BOS/EOS/UNK)
    special_mask = torch.zeros_like(clean_tokens, dtype=torch.bool)
    for sp in special_set:
        special_mask |= (clean_tokens == sp)
    
    # Noise level based on timestep
    noise_level = timesteps.float() / config.N_TIMESTEPS
    
    # Only noise non-special positions
    noise_mask = (torch.rand(B, L, device=device) < noise_level.unsqueeze(1)) & (~special_mask)
    
    # Sample random tokens, avoiding special tokens dynamically
    allowed_tokens = [i for i in range(config.VOCAB_SIZE) if i not in special_set]
    allowed_tensor = torch.tensor(allowed_tokens, device=device)
    
    # Sample from allowed tokens only
    random_indices = torch.randint(0, len(allowed_tokens), (B, L), device=device)
    random_tokens = allowed_tensor[random_indices]
    
    noisy_tokens = clean_tokens.clone()
    noisy_tokens[noise_mask] = random_tokens[noise_mask]
    
    return noisy_tokens, noise_mask  # Return mask for loss computation

def train_step(model, batch, optimizer):
    """Single training step with property-consistency loss."""
    model.train()
    
    clean_tokens = batch['tokens'].to(device)
    conditions = batch['conditions'].to(device)
    use_cond = batch['use_cond'].to(device)
    B = clean_tokens.shape[0]
    
    # Random timesteps
    timesteps = torch.randint(0, config.N_TIMESTEPS, (B,), device=device)
    
    # Add noise to tokens
    noisy_tokens, noise_mask = add_noise_to_tokens(clean_tokens, timesteps)
    
    # Forward pass: predict clean tokens from noisy
    logits, pred_properties = model(noisy_tokens, timesteps, conditions, use_cond)
    
    # Calculate losses
    # 1. Token prediction loss (ONLY on noised positions - prevents identity shortcut)
    flat_logits = logits.reshape(-1, config.VOCAB_SIZE)
    flat_targets = clean_tokens.reshape(-1)
    flat_mask = noise_mask.reshape(-1)
    
    if flat_mask.any():
        masked_logits = flat_logits[flat_mask]
        masked_targets = flat_targets[flat_mask]
        token_loss = criterion(masked_logits, masked_targets)
    else:
        token_loss = criterion(flat_logits, flat_targets)
    
    # 2. Property prediction loss (from noisy input)
    if use_cond.any():
        prop_loss_input = property_criterion(pred_properties[use_cond], conditions[use_cond])
    else:
        prop_loss_input = 0.0 * token_loss
    
    # 3. Property-consistency loss: predict properties from PREDICTED CLEAN tokens
    # This ensures generated molecules actually match target properties (critical!)
    prop_loss_output = torch.tensor(0.0, device=device)
    if use_cond.any():
        # Get distribution over clean tokens from logits
        clean_token_probs = F.softmax(logits, dim=-1)  # [B, L, vocab_size]
        
        # Create soft token embeddings (differentiable)
        token_embeds = model.token_emb.weight  # [vocab_size, d_model]
        soft_embeds = torch.matmul(clean_token_probs, token_embeds)  # [B, L, d_model]
        
        # Pool to get molecule representation
        soft_pooled = soft_embeds.mean(dim=1)  # [B, d_model]
        
        # Predict properties from model's predicted clean tokens
        pred_props_from_output = model.property_head(soft_pooled)
        
        # Loss: predicted properties should match targets
        prop_loss_output = property_criterion(
            pred_props_from_output[use_cond], 
            conditions[use_cond]
        )
    
    # Total loss with MUCH stronger property weighting
    # 0.1 was too weak - model ignored conditioning entirely
    # 1.0 gives equal importance to token accuracy and property control
    total_loss = token_loss + 1.0 * (prop_loss_input + prop_loss_output)
    
    # Backward pass
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    # Combine property losses for logging
    prop_loss_combined = (prop_loss_input.item() if torch.is_tensor(prop_loss_input) else 0.0) + prop_loss_output.item()
    
    return {
        'total_loss': float(total_loss.item()),
        'token_loss': float(token_loss.item()),
        'property_loss': float(prop_loss_combined)  # Combined for training loop compatibility
    }

print("Training functions ready!")

## Training Loop

In [ ]:
# ============================================
# INFERENCE FUNCTIONS (moved before training to enable probes)
# ============================================

def generate_with_cfg(model, target_conditions: Dict[str, float], 
                      num_samples: int = 16, cfg_scale: float = 7.5,
                      max_len: int = 96, num_steps: int = 50,
                      target_only: str = None):
    """
    Generate molecules with Classifier-Free Guidance using discrete diffusion.
    
    Proper denoising: starts with random noise, gradually refines to clean tokens.
    
    Args:
        target_conditions: Dict of target property values
        target_only: If set (e.g. 'qed'), only that property is targeted, others neutral
    """
    model.eval()
    device = next(model.parameters()).device
    
    # Auto-detect single-key target_only mode
    if target_only is None and len(target_conditions) == 1:
        target_only = next(iter(target_conditions.keys()))
    
    # Prepare conditions (target_only mode for QED probes)
    if target_only:
        cond_vec = prepare_conditions(target_conditions, target_only=target_only)
    else:
        cond_vec = prepare_conditions(target_conditions)
    
    cond_vec = cond_vec.unsqueeze(0).repeat(num_samples, 1).to(device)
    
    # Get special token IDs
    pad_id = special_tokens.get('PAD', 0)
    bos_id = special_tokens.get('BOS', 1)
    eos_id = special_tokens.get('EOS', 2)
    
    # Build forbidden set
    special_set = {pad_id, bos_id, eos_id}
    if UNK_ID is not None:
        special_set.add(UNK_ID)
    
    # Initialize with allowed tokens only (no PAD/BOS/EOS/UNK)
    allowed = torch.tensor([i for i in range(config.VOCAB_SIZE) if i not in special_set], device=device)
    x = allowed[torch.randint(0, len(allowed), (num_samples, max_len), device=device)]
    
    # Set BOS at position 0 (never change)
    x[:, 0] = bos_id
    
    # Denoising loop: T → 0 (noisy to clean)
    timesteps = torch.linspace(config.N_TIMESTEPS - 1, 0, num_steps, dtype=torch.long, device=device)
    
    with torch.no_grad():
        for i, t_val in enumerate(tqdm(timesteps, desc="CFG Sampling")):
            # Current timestep for all samples
            t = torch.full((num_samples,), t_val, device=device, dtype=torch.long)
            
            # CFG forward: predict clean token distribution
            guided_logits = model.cfg_forward(x, t, cond_vec, cfg_scale=cfg_scale)
            
            # Block special tokens everywhere except BOS at position 0
            forbidden = [pad_id, bos_id]
            if UNK_ID is not None:
                forbidden.append(UNK_ID)
            
            guided_logits[:, :, forbidden] = -1e9  # Block forbidden tokens
            guided_logits[:, 0, :] = -1e9          # Lock position 0...
            guided_logits[:, 0, bos_id] = 0.0      # ...to BOS only
            
            # CRITICAL: Forbid EOS until minimum length (prevents premature truncation)
            min_length = 20  # Minimum ~20 tokens before allowing EOS
            guided_logits[:, :min_length, eos_id] = -1e9
            
            # Compute noise level (how much to trust prediction vs keep current)
            noise_level = t_val.float() / config.N_TIMESTEPS
            
            if i < len(timesteps) - 1:  # Not the last step
                # Partially denoise: blend prediction with current state
                # Higher noise → trust prediction more
                # Lower noise → keep more of current state
                
                # Sample from predicted distribution with temperature
                # CRITICAL FIX: increase temperature at high noise for exploration
                temp = 1.0 + noise_level * 1.5  # 1.0 (low noise) -> 2.5 (high noise)
                probs = F.softmax(guided_logits / temp, dim=-1)
                x_pred = torch.multinomial(probs.reshape(-1, config.VOCAB_SIZE), 1).reshape(num_samples, max_len)
                
                # Stochastic denoising: gradually replace tokens
                # Use a mask that lets more tokens through as we denoise
                replace_prob = noise_level  # 1.0 → 0.0 (high noise → replace 100%, low noise → replace ~0%)
                replace_mask = torch.rand(num_samples, max_len, device=device) < replace_prob
                
                # Never replace BOS
                replace_mask[:, 0] = False
                
                x = torch.where(replace_mask, x_pred, x)
            else:
                # Final step: greedy decoding (most likely token)
                x = torch.argmax(guided_logits, dim=-1)
            
            # Force BOS at position 0 always
            x[:, 0] = bos_id
    
    # Force EOS at end if none exists (improves decoding stability)
    for i in range(num_samples):
        if not (x[i] == eos_id).any():
            x[i, -1] = eos_id
    
    return x

def tokens_to_molecules(token_sequences):
    """Convert token sequences back to molecules."""
    molecules = []
    
    # Get special token IDs (filter by IDs, not strings!)
    pad_id = special_tokens.get('PAD', 0)
    bos_id = special_tokens.get('BOS', 1)
    eos_id = special_tokens.get('EOS', 2)
    special_ids = {pad_id, bos_id, eos_id}
    if UNK_ID is not None:
        special_ids.add(UNK_ID)
    
    for tokens in token_sequences:
        try:
            # Convert to SELFIES (filter by token IDs, not strings)
            selfies_tokens = []
            for token_id in tokens.cpu().numpy():
                # Stop at EOS FIRST (before skipping specials)
                if token_id == eos_id:
                    break
                # Skip other special tokens by ID
                if token_id in special_ids:
                    continue
                if token_id in id_to_token:
                    selfies_tokens.append(id_to_token[token_id])
                
            selfies_str = ''.join(selfies_tokens)
            
            # Convert to SMILES
            smiles = sf.decoder(selfies_str)
            
            # Validate
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                canonical_smiles = Chem.MolToSmiles(mol)
                properties = calculate_properties(canonical_smiles)
                
                molecules.append({
                    'smiles': canonical_smiles,
                    'selfies': selfies_str,
                    'properties': properties,
                    'valid': True
                })
            else:
                molecules.append({
                    'smiles': None,
                    'selfies': selfies_str,
                    'properties': None,
                    'valid': False
                })
                
        except Exception:
            molecules.append({
                'smiles': None,
                'selfies': None,
                'properties': None,
                'valid': False
            })
    
    return molecules

print("✅ Inference functions defined (generate_with_cfg, tokens_to_molecules)")
print("   Now available for use in probes during training")

In [ ]:
def evaluate_model(model, data_loader):
    """Evaluate model on validation set (no CFG dropout)."""
    model.eval()
    losses = {'total_loss': [], 'token_loss': [], 'property_loss': []}

    with torch.no_grad():
        for batch in data_loader:
            clean_tokens = batch['tokens'].to(device)
            conditions = batch['conditions'].to(device)
            use_cond = batch['use_cond'].to(device)
            B = clean_tokens.size(0)

            # Random timesteps
            t = torch.randint(0, config.N_TIMESTEPS, (B,), device=device)
            
            # Add noise (with mask)
            noisy_tokens, noise_mask = add_noise_to_tokens(clean_tokens, t)

            # Forward pass
            logits, pred_properties = model(noisy_tokens, t, conditions, use_cond)

            # Token loss (ONLY on noised positions - consistent with training)
            flat_logits = logits.reshape(-1, config.VOCAB_SIZE)
            flat_targets = clean_tokens.reshape(-1)
            flat_mask = noise_mask.reshape(-1)

            if flat_mask.any():
                token_loss = criterion(flat_logits[flat_mask], flat_targets[flat_mask])
            else:
                token_loss = criterion(flat_logits, flat_targets)

            # Property loss
            if use_cond.any():
                prop_loss = property_criterion(pred_properties[use_cond], conditions[use_cond])
            else:
                prop_loss = 0.0 * token_loss

            total_loss = token_loss + 0.1 * prop_loss

            losses['total_loss'].append(total_loss.item())
            losses['token_loss'].append(token_loss.item())
            losses['property_loss'].append(prop_loss.item() if torch.is_tensor(prop_loss) else 0.0)

    return {k: float(np.mean(v)) for k, v in losses.items()}

def qed_probe_metrics(model, targets=[0.55, 0.65, 0.75, 0.85], num_samples=20, cfg_scale=7.5):
    """
    Quick probe to check QED conditioning is working.
    
    Returns:
        dict with validity%, achieved QED stats, MAE, Spearman correlation
    """
    from scipy.stats import spearmanr
    
    model.eval()
    results = {'targets': [], 'achieved': []}
    all_valid = []
    
    for target_qed in targets:
        # Neutralize other features (set to 0 in normalized space = mean in raw space)
        test_conditions = {'qed': target_qed}
        
        try:
            # Generate (target_only='qed' ensures other properties are neutral)
            token_seqs = generate_with_cfg(
                model, test_conditions, 
                num_samples=num_samples, 
                cfg_scale=cfg_scale,
                max_len=config.MAX_LEN,
                num_steps=20,  # Fast probe
                target_only='qed'
            )
            
            # Decode and measure
            molecules = tokens_to_molecules(token_seqs)
            valid_mols = [m for m in molecules if m.get('valid', False)]
            
            if valid_mols:
                qed_values = [m['properties']['qed'] for m in valid_mols]
                avg_qed = np.mean(qed_values)
                
                results['targets'].extend([target_qed] * len(qed_values))
                results['achieved'].extend(qed_values)
                all_valid.append(len(valid_mols))
                
                # Show per-target stats for monitoring
                print(f"   Target QED={target_qed:.2f}: achieved={avg_qed:.3f}±{np.std(qed_values):.3f} (n={len(qed_values)})")
            else:
                all_valid.append(0)
                print(f"   Target QED={target_qed:.2f}: no valid molecules")
                
        except Exception as e:
            all_valid.append(0)
            continue
    
    # Compute metrics
    validity_pct = (sum(all_valid) / (len(targets) * num_samples)) * 100 if all_valid else 0.0
    
    if len(results['achieved']) > 0:
        achieved_arr = np.array(results['achieved'])
        targets_arr = np.array(results['targets'])
        mae = np.mean(np.abs(achieved_arr - targets_arr))
        
        # Diagnostic: check variance
        achieved_std = np.std(achieved_arr)
        
        if len(results['achieved']) >= 3:
            # Check for constant values (causes nan in spearmanr)
            if achieved_std < 1e-6:
                spearman = 0.0  # No variance = no correlation
                print(f"   ⚠️ Low variance in predictions (std={achieved_std:.6f}), setting Spearman=0")
            else:
                corr_result = spearmanr(targets_arr, achieved_arr)
                spearman = corr_result.correlation if hasattr(corr_result, 'correlation') else corr_result[0]
                
                # Handle nan explicitly
                if np.isnan(spearman):
                    print(f"   ⚠️ Spearman returned NaN (target_std={np.std(targets_arr):.3f}, pred_std={achieved_std:.3f})")
                    spearman = 0.0
        else:
            spearman = 0.0
    else:
        mae = 999.0
        spearman = 0.0
    
    return {
        'validity_pct': validity_pct,
        'mae': mae,
        'spearman': spearman,
        'n_valid': sum(all_valid),
        'n_total': len(targets) * num_samples,
        'pred_std': achieved_std if len(results['achieved']) > 0 else 0.0  # Added for monitoring
    }

print("Validation and probe functions ready!")
print("   • evaluate_model: computes val loss (no CFG dropout)")
print("   • qed_probe_metrics: tracks conditioning strength early in training")

## Enhanced Diagnostics & Reranking (Option B & C)

In [ ]:
def enhanced_conditioning_diagnostics(model, targets=[0.3, 0.5, 0.7, 0.9], num_samples=30, cfg_scale=7.5):
    """
    Enhanced diagnostics for conditioning quality.
    
    Tests:
    1. Per-target distribution (mean, std, MAE)
    2. Monotonicity (does achieved increase with target?)
    3. Conditional effect size (Δ between max and min targets)  
    4. Diversity (unique SMILES %)
    
    Returns detailed dict with all metrics.
    """
    from scipy.stats import spearmanr
    
    model.eval()
    per_target_results = {}
    all_targets = []
    all_achieved = []
    
    print("\n" + "="*70)
    print("ENHANCED CONDITIONING DIAGNOSTICS")
    print("="*70)
    
    for target_qed in targets:
        test_conditions = {'qed': target_qed}
        
        try:
            # Generate
            token_seqs = generate_with_cfg(
                model, test_conditions,
                num_samples=num_samples,
                cfg_scale=cfg_scale,
                max_len=config.MAX_LEN,
                num_steps=20,
                target_only='qed'
            )
            
            # Decode
            molecules = tokens_to_molecules(token_seqs)
            valid_mols = [m for m in molecules if m.get('valid', False)]
            
            if valid_mols:
                qed_values = [m['properties']['qed'] for m in valid_mols]
                smiles_list = [m['smiles'] for m in valid_mols]
                unique_smiles = set(smiles_list)
                
                # Per-target metrics
                mean_qed = np.mean(qed_values)
                std_qed = np.std(qed_values)
                mae = np.mean(np.abs(np.array(qed_values) - target_qed))
                within_tolerance = np.mean(np.abs(np.array(qed_values) - target_qed) < 0.05) * 100
                
                per_target_results[target_qed] = {
                    'validity_pct': (len(valid_mols) / num_samples) * 100,
                    'mean_qed': mean_qed,
                    'std_qed': std_qed,
                    'mae': mae,
                    'within_tolerance_pct': within_tolerance,
                    'unique_pct': (len(unique_smiles) / len(valid_mols)) * 100
                }
                
                all_targets.extend([target_qed] * len(qed_values))
                all_achieved.extend(qed_values)
                
                # Print per-target
                print(f"\n📊 Target QED = {target_qed:.2f}")
                print(f"   Achieved: {mean_qed:.3f} ± {std_qed:.3f}")
                print(f"   MAE: {mae:.3f} | Within tol: {within_tolerance:.1f}%")
                print(f"   Validity: {per_target_results[target_qed]['validity_pct']:.1f}% | Unique: {per_target_results[target_qed]['unique_pct']:.1f}%")
            else:
                per_target_results[target_qed] = None
                print(f"\n❌ Target QED = {target_qed:.2f}: No valid molecules")
                
        except Exception as e:
            per_target_results[target_qed] = None
            print(f"\n❌ Target QED = {target_qed:.2f}: Error - {str(e)[:50]}")
    
    # Global metrics
    print("\n" + "="*70)
    print("GLOBAL CONDITIONING METRICS")
    print("="*70)
    
    if len(all_achieved) > 0:
        # 1. Overall accuracy
        mae_overall = np.mean(np.abs(np.array(all_achieved) - np.array(all_targets)))
        print(f"\n📈 Overall MAE: {mae_overall:.4f}")
        
        # 2. Spearman correlation
        if len(all_achieved) >= 3:
            spearman, _ = spearmanr(all_targets, all_achieved)
            print(f"📈 Spearman correlation: {spearman:.4f}")
        else:
            spearman = 0.0
            print(f"📈 Spearman correlation: N/A (too few samples)")
        
        # 3. Monotonicity test
        target_means = [per_target_results[t]['mean_qed'] for t in sorted(targets) 
                       if per_target_results[t] is not None]
        if len(target_means) >= 2:
            monotonic = all(target_means[i] <= target_means[i+1] for i in range(len(target_means)-1))
            print(f"📈 Monotonicity: {'✅ PASS' if monotonic else '❌ FAIL'}")
            if not monotonic:
                print(f"   Achieved means: {[f'{m:.3f}' for m in target_means]}")
        
        # 4. Conditional effect size (Δ)
        valid_targets = [t for t in targets if per_target_results[t] is not None]
        if len(valid_targets) >= 2:
            min_target = min(valid_targets)
            max_target = max(valid_targets)
            delta = per_target_results[max_target]['mean_qed'] - per_target_results[min_target]['mean_qed']
            expected_delta = max_target - min_target
            print(f"📈 Conditional effect size (Δ): {delta:.4f} (expected: {expected_delta:.4f})")
            
            if delta < 0.05:
                print(f"   ⚠️  WARNING:  Δ is very small - conditioning may be ignored!")
            elif delta < expected_delta * 0.3:
                print(f"   ⚠️  WARNING: Δ is much smaller than expected - weak conditioning")
            else:
                print(f"   ✅ Δ is reasonable - conditioning is working")
        
    else:
        mae_overall = 999.0
        spearman = 0.0
        print("\n❌ No valid molecules generated - cannot compute metrics")
    
    print("="*70 + "\n")
    
    return {
        'per_target': per_target_results,
        'mae_overall': mae_overall,
        'spearman': spearman,
        'all_targets': all_targets,
        'all_achieved': all_achieved
    }

print("✅ Enhanced diagnostics function ready")

In [ ]:
def generate_many_and_rerank(model, target_conditions, 
                              num_candidates=256, top_k=16,
                              cfg_scale=7.5, diversity_threshold=0.6):
    """
    Option C: Generate-many + rerank with diversity filter.
    
    Strategy:
    1. Generate N candidates (default 256)
    2. Score by property closeness to target
    3. Keep top K (default 16)
    4. Apply diversity filter (Tanimoto threshold)
    
    Args:
        model: trained model
        target_conditions: dict like {'qed': 0.7, 'mw': 300, ...}
        num_candidates: number of molecules to generate
        top_k: how many to keep after scoring
        cfg_scale: CFG guidance strength
        diversity_threshold: min Tanimoto distance for diversity filter
    
    Returns:
        dict with best molecules, scores, and statistics
    """
    from rdkit import Chem, DataStructs
    from rdkit.Chem import AllChem
    
    print(f"\n{'='*70}")
    print(f"GENERATE-MANY + RERANK (Option C)")
    print(f"{'='*70}")
    print(f"Generating {num_candidates} candidates...")
    
    model.eval()
    
    # Generate candidates in batches (to avoid OOM)
    batch_size = 64
    all_molecules = []
    
    for batch_start in range(0, num_candidates, batch_size):
        batch_num = min(batch_size, num_candidates - batch_start)
        
        try:
            token_seqs = generate_with_cfg(
                model, target_conditions,
                num_samples=batch_num,
                cfg_scale=cfg_scale,
                max_len=config.MAX_LEN,
                num_steps=20
            )
            
            molecules = tokens_to_molecules(token_seqs)
            all_molecules.extend(molecules)
            
        except Exception as e:
            print(f"   ⚠️ Batch {batch_start // batch_size + 1} failed: {str(e)[:50]}")
            continue
    
    # Filter valid molecules
    valid_mols = [m for m in all_molecules if m.get('valid', False)]
    print(f"✅ Generated: {len(valid_mols)}/{num_candidates} valid ({len(valid_mols)/num_candidates*100:.1f}%)")
    
    if len(valid_mols) == 0:
        print("❌ No valid molecules generated!")
        return None
    
    # Score by property closeness
    print(f"\n📊 Scoring by property closeness...")
    target_qed = target_conditions.get('qed', 0.7)
    
    for mol in valid_mols:
        achieved_qed = mol['properties']['qed']
        error = abs(achieved_qed - target_qed)
        mol['score'] = -error  # Higher score = closer to target
        mol['error'] = error
    
    # Sort by score
    valid_mols.sort(key=lambda x: x['score'], reverse=True)
    
    # Keep top K
    top_mols = valid_mols[:top_k]
    print(f"✅ Kept top {len(top_mols)} by score")
    
    # Diversity filter using Tanimoto
    print(f"\n🎨 Applying diversity filter (Tanimoto threshold={diversity_threshold})...")
    diverse_mols = []
    fingerprints = []
    
    for mol in top_mols:
        try:
            rdkit_mol = Chem.MolFromSmiles(mol['smiles'])
            fp = AllChem.GetMorganFingerprintAsBitVect(rdkit_mol, 2, nBits=2048)
            
            # Check similarity to already selected molecules
            is_diverse = True
            for existing_fp in fingerprints:
                similarity = DataStructs.TanimotoSimilarity(fp, existing_fp)
                if similarity > diversity_threshold:
                    is_diverse = False
                    break
            
            if is_diverse or len(diverse_mols) == 0:  # Always keep first molecule
                diverse_mols.append(mol)
                fingerprints.append(fp)
                
        except Exception:
            continue
    
    print(f"✅ Diverse set: {len(diverse_mols)} molecules")
    
    # Statistics
    diverse_qeds = [m['properties']['qed'] for m in diverse_mols]
    diverse_errors = [m['error'] for m in diverse_mols]
    
    print(f"\n{'='*70}")
    print(f"RERANKING RESULTS")
    print(f"{'='*70}")
    print(f"Target QED: {target_qed:.3f}")
    print(f"Best achieved: {diverse_qeds[0]:.3f} (error: {diverse_errors[0]:.4f})")
    print(f"Mean achieved: {np.mean(diverse_qeds):.3f} ± {np.std(diverse_qeds):.3f}")
    print(f"MAE: {np.mean(diverse_errors):.4f}")
    print(f"Unique SMILES: {len(set([m['smiles'] for m in diverse_mols]))}/{len(diverse_mols)}")
    
    # Show top 5
    print(f"\n🏆 Top 5 molecules:")
    for i, mol in enumerate(diverse_mols[:5]):
        print(f"   {i+1}. QED={mol['properties']['qed']:.3f} (Δ={mol['error']:.4f})")
        print(f"      {mol['smiles'][:80]}")
    
    print(f"{'='*70}\n")
    
    return {
        'diverse_molecules': diverse_mols,
        'all_valid': valid_mols,
        'target_qed': target_qed,
        'best_qed': diverse_qeds[0],
        'mean_qed': np.mean(diverse_qeds),
        'mae': np.mean(diverse_errors),
        'num_unique': len(set([m['smiles'] for m in diverse_mols]))
    }

print("✅ Reranking function ready (Option C)")

In [ ]:
# Training history
train_history = {
    'epoch': [],
    'train_total_loss': [],
    'train_token_loss': [], 
    'train_property_loss': [],
    'val_total_loss': [],
    'learning_rate': [],
    # QED probe metrics
    'probe_validity': [],
    'probe_mae': [],
    'probe_spearman': []
}

# Best model tracking (based on validation loss)
best_val_loss = float('inf')
best_epoch = 0
best_model_path = os.path.join(config.MODEL_DIR, 'strong_diffusion_best.pt')

# Early stopping configuration
early_stop_patience = 15  # Stop if no improvement for 15 epochs
early_stop_counter = 0
min_improvement = 1e-4  # Minimum improvement threshold

print("Starting training...")
print(f"   Epochs: {config.EPOCHS}")
print(f"   Batch size: {config.BATCH_SIZE}")
print(f"   Condition dropout: {config.COND_DROPOUT} (for CFG)")
print(f"   Early stopping patience: {early_stop_patience} epochs")
print("="*70)

for epoch in range(config.EPOCHS):
    start_time = time.time()
    epoch_losses = {'total_loss': [], 'token_loss': [], 'property_loss': []}
    
    # Training loop
    model.train()
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.EPOCHS}")
    
    for batch_idx, batch in enumerate(progress_bar):
        step_losses = train_step(model, batch, optimizer)
        
        # Accumulate losses
        for key in epoch_losses:
            epoch_losses[key].append(step_losses[key])
        
        # Update progress bar
        if batch_idx % config.LOG_EVERY == 0:
            current_lr = optimizer.param_groups[0]['lr']
            progress_bar.set_postfix({
                'Loss': f"{step_losses['total_loss']:.4f}",
                'Token': f"{step_losses['token_loss']:.4f}",
                'Prop': f"{step_losses['property_loss']:.4f}",
                'LR': f"{current_lr:.2e}"
            })
    
    # End of epoch
    scheduler.step()
    epoch_time = time.time() - start_time
    
    # Calculate average training losses
    avg_train_losses = {key: np.mean(values) for key, values in epoch_losses.items()}
    current_lr = optimizer.param_groups[0]['lr']
    
    # Evaluate on validation set (no CFG dropout)
    print(f"\nEpoch {epoch+1}/{config.EPOCHS} - Evaluating...")
    avg_val_losses = evaluate_model(model, val_loader)
    
    # QED probe metrics (quick conditioning check)
    probe_metrics = qed_probe_metrics(model, targets=[0.55, 0.65, 0.75, 0.85], num_samples=20, cfg_scale=7.5)
    
    # Log epoch results
    print(f"\nEpoch {epoch+1}/{config.EPOCHS} ({epoch_time:.1f}s)")
    print(f"   Train Loss: {avg_train_losses['total_loss']:.4f} | Val Loss: {avg_val_losses['total_loss']:.4f}")
    print(f"   Token: {avg_train_losses['token_loss']:.4f} | Prop: {avg_train_losses['property_loss']:.4f}")
    print(f"   Learning Rate: {current_lr:.2e}")
    print(f"   QED Probe: Valid={probe_metrics['validity_pct']:.1f}% | MAE={probe_metrics['mae']:.3f} | Spearman={probe_metrics['spearman']:.3f}")
    
    # Update training history
    train_history['epoch'].append(epoch + 1)
    train_history['train_total_loss'].append(avg_train_losses['total_loss'])
    train_history['train_token_loss'].append(avg_train_losses['token_loss'])
    train_history['train_property_loss'].append(avg_train_losses['property_loss'])
    train_history['val_total_loss'].append(avg_val_losses['total_loss'])
    train_history['learning_rate'].append(current_lr)
    train_history['probe_validity'].append(probe_metrics['validity_pct'])
    train_history['probe_mae'].append(probe_metrics['mae'])
    train_history['probe_spearman'].append(probe_metrics['spearman'])
    
    # Save best model (based on VALIDATION loss)
    if avg_val_losses['total_loss'] < best_val_loss - min_improvement:
        best_val_loss = avg_val_losses['total_loss']
        best_epoch = epoch + 1
        early_stop_counter = 0  # Reset counter on improvement
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_losses': avg_train_losses,
            'val_losses': avg_val_losses,
            'best_val_loss': best_val_loss,
            'probe_metrics': probe_metrics
        }, best_model_path)
        
        print(f"✅ NEW BEST! Val loss: {best_val_loss:.4f} (Spearman: {probe_metrics['spearman']:.3f})")
    else:
        early_stop_counter += 1
        print(f"⚠️  No improvement: {early_stop_counter}/{early_stop_patience}")
    
    # Save periodic checkpoint
    if (epoch + 1) % config.SAVE_EVERY == 0:
        checkpoint_path = os.path.join(config.MODEL_DIR, f'strong_diffusion_epoch{epoch+1}.pt')
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_losses': avg_train_losses,
            'val_losses': avg_val_losses
        }, checkpoint_path)
        
        print(f"Saved checkpoint: {checkpoint_path}")
    
    # Check if early stopping triggered
    if early_stop_counter >= early_stop_patience:
        print(f"\n🛑 Early stopping triggered! No improvement for {early_stop_patience} epochs")
        print(f"Best model: Epoch {best_epoch} with val loss {best_val_loss:.4f}")
        break

print("\n" + "="*70)
print("🎉 TRAINING COMPLETED!")
print("="*70)
print(f"Final QED probe: Spearman={train_history['probe_spearman'][-1]:.3f}, MAE={train_history['probe_mae'][-1]:.3f}")
print(f"Best model: Epoch {best_epoch} with val loss {best_val_loss:.4f}")
print(f"Best model saved: {best_model_path}")

## 📥 Load Checkpoint (After Kernel Restart)

If you restarted the kernel, run this cell to load your trained model.

In [ ]:
# ============================================
# LOAD TRAINED MODEL FROM CHECKPOINT
# ============================================
# Run this after kernel restart to load your trained model

# 1. Reinitialize model architecture
model = StrongConditionalDiffusion(
    vocab_size=config.VOCAB_SIZE,
    max_len=config.MAX_LEN,
    d_model=config.D_MODEL,
    n_heads=config.N_HEADS,
    n_layers=config.N_LAYERS,
    dropout=config.DROPOUT,
    n_conditions=config.N_CONDITIONS,
    cond_dropout=config.COND_DROPOUT
).to(device)

print(f"Model initialized: {sum(p.numel() for p in model.parameters()):,} parameters")

# 2. Load checkpoint
best_model_path = os.path.join(config.MODEL_DIR, 'strong_diffusion_best.pt')

if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()  # Set to eval mode for inference
    
    print(f"\n✅ Loaded checkpoint from epoch {checkpoint['epoch']}")
    print(f"   Val Loss: {checkpoint['best_val_loss']:.4f}")
    print(f"   Probe Metrics:")
    
    # Handle different possible key names
    probe = checkpoint.get('probe_metrics', {})
    if 'valid_pct' in probe:
        print(f"      Valid: {probe['valid_pct']:.1f}%")
    elif 'validity' in probe:
        print(f"      Valid: {probe['validity']:.1f}%")
    
    if 'mae' in probe:
        print(f"      MAE: {probe['mae']:.4f}")
    if 'spearman' in probe:
        print(f"      Spearman: {probe['spearman']:.3f}")
    
    print(f"\nModel ready for inference! 🚀")
else:
    print(f"❌ Checkpoint not found at: {best_model_path}")
    print("   Make sure you've run the training loop first.")

In [ ]:
# ============================================
# LOAD CONDITION STATISTICS
# ============================================
# Required for proper normalization during inference

import json

stats_path = os.path.join(config.MODEL_DIR, 'condition_stats.json')

if os.path.exists(stats_path):
    with open(stats_path, 'r') as f:
        cond_stats = json.load(f)
    print(f"✅ Loaded condition statistics from: {stats_path}")
    print(f"   Features: {list(cond_stats.keys())}")
else:
    print(f"❌ Stats file not found: {stats_path}")
    print("   Cannot run diagnostics without condition stats!")

## Post-Training Analysis

**Instructions:** Run the cells below AFTER your model has finished training to:
1. **Diagnostics**: Check conditioning quality (monotonicity, effect size, diversity)
2. **Reranking**: Generate high-quality molecules with diversity filtering

In [ ]:
# ============================================
# 1. ENHANCED DIAGNOSTICS
# ============================================
# Tests conditioning quality across multiple targets

diagnostics = enhanced_conditioning_diagnostics(
    model, 
    targets=[0.3, 0.5, 0.7, 0.9],  # Test full QED spectrum
    num_samples=30,
    cfg_scale=7.5
)

# Check results
print("\n📊 Summary:")
print(f"Overall MAE: {diagnostics['mae_overall']:.4f}")
print(f"Spearman: {diagnostics['spearman']:.4f}")

In [ ]:
# ============================================
# 2. GENERATE WITH RERANKING
# ============================================
# Generate 256 candidates, keep top diverse molecules

results = generate_many_and_rerank(
    model,
    target_conditions={'qed': 0.85},  # Target high druglikeness
    num_candidates=256,
    top_k=32,
    cfg_scale=7.5,
    diversity_threshold=0.6
)

# Display best molecules
if results:
    print("\n🏆 TOP 10 BEST MOLECULES")
    print("="*70)
    
    best_mols = results['diverse_molecules'][:10]
    for i, mol in enumerate(best_mols):
        print(f"\n{i+1}. QED: {mol['properties']['qed']:.3f} (Δ={mol['error']:.4f})")
        print(f"   {mol['smiles']}")
    
    print("\n" + "="*70)
    print("📊 SUMMARY")
    print("="*70)
    print(f"Target QED:       {results['target_qed']:.3f}")
    print(f"Best achieved:    {results['best_qed']:.3f}")
    print(f"Mean achieved:    {results['mean_qed']:.3f} ± {np.std([m['properties']['qed'] for m in best_mols]):.3f}")
    print(f"MAE:              {results['mae']:.4f}")
    print(f"Unique molecules: {results['num_unique']}/{len(best_mols)}")
else:
    print("❌ No valid molecules generated")

In [ ]:
# ============================================
# 3. TEST MULTIPLE TARGETS (Optional)
# ============================================
# Compare reranking performance across different QED targets

test_targets = [0.5, 0.7, 0.9]
comparison_results = []

for target_qed in test_targets:
    print(f"\n{'='*70}")
    print(f"Testing target QED = {target_qed}")
    print('='*70)
    
    result = generate_many_and_rerank(
        model,
        target_conditions={'qed': target_qed},
        num_candidates=128,  # Smaller for faster testing
        top_k=16,
        cfg_scale=7.5,
        diversity_threshold=0.6
    )
    
    if result:
        comparison_results.append({
            'target': target_qed,
            'best': result['best_qed'],
            'mean': result['mean_qed'],
            'mae': result['mae']
        })

# Summary comparison
if comparison_results:
    print("\n" + "="*70)
    print("COMPARISON ACROSS TARGETS")
    print("="*70)
    print(f"{'Target':<10} {'Best':<10} {'Mean':<10} {'MAE':<10}")
    print("-"*70)
    for r in comparison_results:
        print(f"{r['target']:<10.2f} {r['best']:<10.3f} {r['mean']:<10.3f} {r['mae']:<10.4f}")

## Training Visualization

In [ ]:
def _as_list(x):
    # supports list, numpy, pandas Series
    try:
        return list(x)
    except Exception:
        return [x]

def _get(train_history, key, default=None):
    if isinstance(train_history, dict):
        return train_history.get(key, default)
    # pandas DataFrame-like
    try:
        return train_history[key]
    except Exception:
        return default

def plot_training_curves(train_history, best_epoch=None, best_val_loss=None, title="Strong Conditional Diffusion Training Progress"):
    # ---- Extract epochs (robust) ----
    epochs = _get(train_history, "epoch", None)
    if epochs is None:
        # infer from any available metric length
        for k in ["train_total_loss", "val_total_loss", "learning_rate"]:
            v = _get(train_history, k, None)
            if v is not None:
                epochs = list(range(1, len(_as_list(v)) + 1))
                break
    epochs = _as_list(epochs)

    # ---- Extract metrics (may be missing) ----
    train_total = _get(train_history, "train_total_loss", None)
    val_total   = _get(train_history, "val_total_loss", None)
    lr          = _get(train_history, "learning_rate", None)

    token_loss  = _get(train_history, "train_token_loss", None)
    prop_loss   = _get(train_history, "train_property_loss", None)

    validity    = _get(train_history, "probe_validity", None)
    probe_mae   = _get(train_history, "probe_mae", None)
    spearman    = _get(train_history, "probe_spearman", None)

    # ---- Compute best epoch/loss if not provided ----
    if best_val_loss is None or best_epoch is None:
        if val_total is not None:
            val_arr = np.asarray(_as_list(val_total), dtype=float)
            best_idx = int(np.nanargmin(val_arr))
            best_val_loss = float(val_arr[best_idx])
            best_epoch = epochs[best_idx] if best_idx < len(epochs) else best_idx + 1
        else:
            best_val_loss, best_epoch = np.nan, None

    # ---- Helpers for plotting safely ----
    def plot_if(ax, y, *args, **kwargs):
        if y is None:
            ax.text(0.5, 0.5, "Metric missing", ha="center", va="center", alpha=0.6, transform=ax.transAxes)
            ax.set_xticks([])
            ax.set_yticks([])
            return False
        y = _as_list(y)
        ax.plot(epochs[:len(y)], y, *args, **kwargs)
        return True

    # ---- Figure ----
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

    # Row 1: Total Loss
    ax1 = fig.add_subplot(gs[0, :2])
    had_train = plot_if(ax1, train_total, "b-", linewidth=2, label="Train")
    had_val   = plot_if(ax1, val_total,   "r-", linewidth=2, label="Validation")
    ax1.set_title("Total Loss (Train vs Val)", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    if had_train or had_val:
        ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Learning rate
    ax2 = fig.add_subplot(gs[0, 2])
    had_lr = plot_if(ax2, lr, color="purple", linewidth=2)
    ax2.set_title("Learning Rate", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("LR")
    if had_lr:
        ax2.set_yscale("log")
    ax2.grid(True, alpha=0.3)

    # Row 2: Component losses
    ax3 = fig.add_subplot(gs[1, 0])
    plot_if(ax3, token_loss, "g-", linewidth=2)
    ax3.set_title("Token Loss", fontsize=14, fontweight="bold")
    ax3.set_xlabel("Epoch")
    ax3.set_ylabel("Loss")
    ax3.grid(True, alpha=0.3)

    ax4 = fig.add_subplot(gs[1, 1])
    plot_if(ax4, prop_loss, color="orange", linewidth=2)
    ax4.set_title("Property Loss", fontsize=14, fontweight="bold")
    ax4.set_xlabel("Epoch")
    ax4.set_ylabel("Loss")
    ax4.grid(True, alpha=0.3)

    # Validity %
    ax5 = fig.add_subplot(gs[1, 2])
    plot_if(ax5, validity, color="cyan", linewidth=2)
    ax5.set_title("Validity % (Probe)", fontsize=14, fontweight="bold")
    ax5.set_xlabel("Epoch")
    ax5.set_ylabel("%")
    ax5.grid(True, alpha=0.3)

    # Row 3: Probe metrics
    ax6 = fig.add_subplot(gs[2, 0])
    had_mae = plot_if(ax6, probe_mae, "m-", linewidth=2)
    ax6.set_title("QED MAE (Probe)", fontsize=14, fontweight="bold")
    ax6.set_xlabel("Epoch")
    ax6.set_ylabel("MAE")
    if had_mae:
        ax6.axhline(y=0.05, color="r", linestyle="--", label="Target (0.05)")
        ax6.legend()
    ax6.grid(True, alpha=0.3)

    ax7 = fig.add_subplot(gs[2, 1])
    had_spear = plot_if(ax7, spearman, color="darkgreen", linewidth=2)
    ax7.set_title("Spearman Correlation (Probe)", fontsize=14, fontweight="bold")
    ax7.set_xlabel("Epoch")
    ax7.set_ylabel("Spearman ρ")
    if had_spear:
        ax7.axhline(y=0.0, color="gray", linestyle="--", alpha=0.5)
        ax7.axhline(y=0.9, color="g", linestyle="--", label="Target (>0.9)")
        ax7.legend()
    ax7.grid(True, alpha=0.3)

    # Summary text
    ax8 = fig.add_subplot(gs[2, 2])
    ax8.axis("off")

    def last_val(x):
        if x is None:
            return None
        x = _as_list(x)
        return x[-1] if len(x) else None

    v_last = last_val(val_total)
    t_last = last_val(train_total)
    valid_last = last_val(validity)
    mae_last = last_val(probe_mae)
    spear_last = last_val(spearman)

    summary_text = (
        "Final Metrics:\n\n"
        f"Val Loss:   {v_last:.4f}\n" if v_last is not None else "Val Loss:   N/A\n"
    )
    summary_text += (
        f"Train Loss: {t_last:.4f}\n\n" if t_last is not None else "Train Loss: N/A\n\n"
    )
    summary_text += "QED Probe (Final):\n"
    summary_text += f"  Validity: {valid_last:.1f}%\n" if valid_last is not None else "  Validity: N/A\n"
    summary_text += f"  MAE:      {mae_last:.3f}\n" if mae_last is not None else "  MAE:      N/A\n"
    summary_text += f"  Spearman: {spear_last:.3f}\n\n" if spear_last is not None else "  Spearman: N/A\n\n"
    summary_text += f"Best Epoch: {best_epoch}\n"
    summary_text += f"Best Val Loss: {best_val_loss:.4f}\n\n"
    summary_text += "✅ Spearman > 0 indicates\n   conditioning is working!\n"

    ax8.text(
        0.05, 0.5, summary_text,
        transform=ax8.transAxes,
        fontsize=11, va="center", family="monospace",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.3)
    )

    plt.suptitle(title, fontsize=16, fontweight="bold")
    plt.show()

    # ---- Print final stats (robust) ----
    final_epoch = epochs[-1] if len(epochs) else None
    print("\n" + "=" * 70)
    print(f"FINAL RESULTS (Epoch {final_epoch})")
    print("=" * 70)
    print(f"Train Loss: {t_last:.4f}" if t_last is not None else "Train Loss: N/A")
    print(f"Val Loss:   {v_last:.4f}" if v_last is not None else "Val Loss:   N/A")
    print(f"Best Val Loss: {best_val_loss:.4f} (Epoch {best_epoch})" if best_epoch is not None else f"Best Val Loss: {best_val_loss:.4f}")
    print("\nQED Probe Metrics:")
    print(f"  Validity: {valid_last:.1f}%" if valid_last is not None else "  Validity: N/A")
    print(f"  MAE:      {mae_last:.3f} (target: <0.05)" if mae_last is not None else "  MAE:      N/A (target: <0.05)")
    print(f"  Spearman: {spear_last:.3f} (target: >0.9)" if spear_last is not None else "  Spearman: N/A (target: >0.9)")
    print("=" * 70)

# Usage:
# plot_training_curves(train_history)
# or plot_training_curves(train_history, best_epoch=best_epoch, best_val_loss=best_val_loss)


## Conditioning Diagnostics

In [ ]:
# ============================================
# CONDITIONING DIAGNOSTICS
# ============================================
# Test if conditioning actually works with monotonicity check

def diagnose_conditioning(model, property_range: Tuple[float, float] = (0.3, 0.9),
                         num_points: int = 7, num_samples_per_point: int = 8):
    """
    Verify conditioning monotonicity: higher target QED -> higher generated QED
    
    Also tracks logit differences: |logits_cond - logits_uncond|
    If this is tiny, conditioning is being ignored!
    """
    model.eval()
    device = next(model.parameters()).device
    
    targets = np.linspace(property_range[0], property_range[1], num_points)
    results = {'target': [], 'generated': [], 'logit_diff': []}
    
    print("Running conditioning diagnostics...")
    print("   Testing QED monotonicity across target range\n")
    
    # Create a reference input for logit diff calculation
    ref_tokens = torch.randint(0, config.VOCAB_SIZE, (1, config.MAX_LEN), device=device)
    ref_t = torch.tensor([config.N_TIMESTEPS // 2], device=device)  # Mid-timestep
    
    for target_qed in tqdm(targets, desc="Testing targets"):
        # Generate molecules for this target
        generated_qed_values = []
        logit_diffs = []
        
        for _ in range(num_samples_per_point):
            try:
                # Use target_only to only set QED, others neutral
                conditions = prepare_conditions({'qed': target_qed}, target_only='qed')
                conditions = conditions.unsqueeze(0).to(device)
                
                # Calculate logit difference (conditioning strength indicator)
                with torch.no_grad():
                    cond_logits, _ = model(ref_tokens, ref_t, conditions)
                    uncond_logits, _ = model(ref_tokens, ref_t, None)
                    
                    diff = (cond_logits - uncond_logits).abs().mean().item()
                    logit_diffs.append(diff)
                
                # Generate and measure actual QED
                token_seqs = generate_with_cfg(
                    model, {'qed': target_qed}, 
                    num_samples=1, cfg_scale=7.5, 
                    max_len=config.MAX_LEN,
                    target_only='qed'
                )
                
                # Decode tokens to molecules
                molecules = tokens_to_molecules(token_seqs)
                for mol in molecules:
                    if mol.get('valid', False):
                        generated_qed_values.append(mol['properties']['qed'])
                        
            except Exception as e:
                continue
        
        if generated_qed_values:
            avg_qed = np.mean(generated_qed_values)
            avg_diff = np.mean(logit_diffs)
            results['target'].append(target_qed)
            results['generated'].append(avg_qed)
            results['logit_diff'].append(avg_diff)
            
            print(f"   Target QED: {target_qed:.2f} -> Generated: {avg_qed:.2f} (diff: {avg_qed - target_qed:+.2f}), Logit Delta: {avg_diff:.3f}")
    
    # Check monotonicity
    if len(results['generated']) >= 3:
        correlation = np.corrcoef(results['target'], results['generated'])[0, 1]
        is_monotonic = correlation > 0.5
        
        print(f"\nDIAGNOSIS RESULTS:")
        print(f"   Correlation (target vs generated): {correlation:.3f}")
        print(f"   Monotonicity check: {'PASS' if is_monotonic else 'FAIL'}")
        print(f"   Avg logit difference: {np.mean(results['logit_diff']):.3f}")
        
        if np.mean(results['logit_diff']) < 0.1:
            print("   WARNING: Logit diff is small - conditioning may be ignored!")
        
        return results, correlation, is_monotonic
    
    return results, None, False

# Run diagnostics (optional - uncomment to run after training)
# diagnostics_results, corr, monotonic = diagnose_conditioning(model)

print("Diagnostics function ready - run diagnose_conditioning(model) to test")


## Save Final Model

In [ ]:
# Save final trained model
final_model_path = os.path.join(config.MODEL_DIR, 'strong_conditional_diffusion_final.pt')

final_checkpoint = {
    'model_state_dict': model.state_dict(),
    'config': {k: getattr(config, k) for k in dir(config) if k.isupper()},  # Save as dict, not class instance
    'train_history': train_history,
    'total_params': total_params,
    'vocab_size': config.VOCAB_SIZE,
    'special_tokens': special_tokens,
    'condition_stats': cond_stats,
    'final_performance': {
        'epoch': train_history['epoch'][-1],
        'train_total_loss': train_history['train_total_loss'][-1],
        'val_total_loss': train_history['val_total_loss'][-1],
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'probe_validity': train_history['probe_validity'][-1],
        'probe_mae': train_history['probe_mae'][-1],
        'probe_spearman': train_history['probe_spearman'][-1]
    }
}

torch.save(final_checkpoint, final_model_path)

print(f"Final model saved: {final_model_path}")
print(f"Model summary:")
print(f"   Parameters: {total_params:,}")
print(f"   Final train loss: {train_history['train_total_loss'][-1]:.4f}")
print(f"   Final val loss: {train_history['val_total_loss'][-1]:.4f}")
print(f"   Best val loss: {best_val_loss:.4f} (epoch {best_epoch})")
print(f"   Training epochs: {len(train_history['epoch'])}")
print(f"   Final Spearman: {train_history['probe_spearman'][-1]:.3f}")

# Save training history as JSON for analysis
history_path = os.path.join(config.MODEL_DIR, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(train_history, f, indent=2)

print(f"\nTraining history saved: {history_path}")
print(f"{'='*70}")

## Summary & Next Steps

**Strong Conditional Diffusion Training Completed!**

### What We Built:
- **Cross-Attention Conditioning**: Every transformer layer can "see" target properties
- **Classifier-Free Guidance**: 15% condition dropout enables CFG amplification  
- **Multi-Scale Embeddings**: Rich condition representation with learnable tokens
- **CFG Inference**: Configurable guidance scale for precise control

### Expected Performance:
- **Current weak model**: +/-0.15 QED error
- **This architecture**: +/-0.02-0.05 QED error (3-7x better!)

### Key Innovation:
The **cross-attention mechanism** allows tokens to continuously "look at" target properties during generation, providing the strong conditional control that was missing in the original architecture.